In [11]:
import json
import httpx
from parsel import Selector
from bs4 import BeautifulSoup
import requests
import webbrowser

In [12]:
session = httpx.Client(
    headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36 Edg/113.0.1774.35",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
    },
    http2=True,
    follow_redirects=True
)

In [13]:
response = session.get("https://www.ebay.com/sch/i.html?_dcat=183473&_fsrp=1&rt=nc&_from=R40&_nkw=warhammer40k&_sacat=0&LH_TitleDesc=0&Army=Adepta%2520Sororitas%7CChaos%2520Space%2520Marines%7CDeath%2520Guard%7CTyranids%7CImperial%2520Knights")
print(response)

<Response [200 OK]>


In [14]:
soup = BeautifulSoup(response, 'html.parser')
print(soup)

<!DOCTYPE html>
<!--[if IE 9]><html class="ie9" lang="en"><![endif]--><!--[if gt IE 9]><!--><html lang="en"><!--<![endif]--><!--M_fbd99063#s0-2-1--><noscript class="x-page-config"></noscript><!--M_fbd99063/--><head><!--M_fbd99063#s0-2-4--><meta content="IE=edge" http-equiv="X-UA-Compatible"/><!-- SEO METADATA START --><meta content="website" property="og:type"><link href="https://www.ebay.com/sch/i.html?_nkw=warhammer40k&amp;_sop=12" rel="canonical"/><meta content="Get the best deals for Warhammer40k at eBay.com. We have a great online selection at the lowest prices with Fast &amp; Free shipping on many items!" name="description"/><meta content="https://ir.ebaystatic.com/cr/v/c1/ebay-logo-1-1200x630-margin.png" property="og:image"/><meta content="102628213125203" property="fb:app_id"/><meta content="sites-7757056108965234" name="google-adsense-account"/><meta content="eBay" property="og:site_name"/><link href="https://i.ebayimg.com" rel="preconnect"/><meta content="unsafe-url" name="re

In [15]:
listings = soup.find('ul', class_='srp-results srp-grid clearfix').select('li', class_='s-card s-card--vertical s-card--dark-solt-links-black')

In [16]:
print(listings)

[<li class="s-card s-card--vertical" data-gr2="2" data-gr3="2" data-gr4="2" data-listingid="188127069385" data-view="mi:1686|iid:1" data-viewport='{"trackableId":"01KKACC89CP1NVETBTJGQMP0KJ"}' id="item2bcd3f44c9"><div class="su-card-container su-card-container--vertical"><div class="su-card-container__media"><div class="s-card__media-wrapper"><!--M_fbd99063#s0-2-54-0-9-8-4-4-0-3-0-4[0]-1-0-1-1-1-2-5--><div class="su-media-multi-image"><div class="su-media-multi-image__primary"><div class="su-image"><a _sp="p2351460.m1686.l7400" class="s-card__link image-treatment" data-interactions='[{"actionKind":"NAVSRC","interaction":"wwFVrK2vRE0lhQQ0MDFLS0FDQzhDODgwN0g2OUFBUDQwVlZNQ0Y0MDFLS0FDQzg5Q1AxTlZFVEJUSkdRTVAwS0oAAAg3NDAwDE5BVlNSQwA="}]' data-s-d6zx939='{"eventFamily":"LST","eventAction":"ACTN","actionKind":"NAVSRC","actionKinds":["NAVSRC"],"operationId":"2351460","flushImmediately":false,"eventProperty":{"$l":"3330152042516216"}}' href="https://www.ebay.com/itm/188127069385?_skw=warhammer40

In [17]:
images = []
for listing in listings:
    images.append(listing.find('img', class_='s-card__image'))
prices = []
for listing in listings:
    try:
        pricecontainer = listing.find('div', class_='s-card__attribute-row')
        price = pricecontainer.select('span', class_='su-styled-text primary bold large-1 s-card__price')
        for i in range(0, len(price)):
            price[i] = price[i].text
        prices.append(''.join(price))
    except AttributeError:
        prices.append('Unknown price')

In [18]:
displaydata = list(zip(listings, images, prices))

In [19]:
html_content = ''
html_array = []

for item in displaydata:
    try:
        link = item[0].find('a')['href']
        name = item[0].find('span', class_='su-styled-text primary default')
        image = '<img src=' + item[1]['src'] + ' width=500>'
        if not name or name.text == 'Shop on eBay':
            raise AttributeError()
        html_content += image
        html_content += '<br>'
        html_content += '<a href=' + link + '>'+name.text+'</a>'
        html_content += '<br>'
        html_content += '<span>' + item[2] + '</span>'
        html_content += '<br>'
        html_content += '<br>'
        html_item = [image, '<a href=' + link + '>'+name.text+'</a>', '<span>' + item[2] + '</span>']
        html_array.append(html_item)
    except AttributeError:
        pass
    except TypeError:
        pass
    except KeyError:
        pass
f = open('output.html', 'w')
f.write(html_content)
f.close()
print(html_array)

[['<img src=https://i.ebayimg.com/images/g/NsoAAeSw-Llpqtag/s-l500.webp width=500>', '<a href=https://www.ebay.com/itm/188127069385?_skw=warhammer40k&itmmeta=01KKACC89CP1NVETBTJGQMP0KJ&hash=item2bcd3f44c9:g:NsoAAeSw-Llpqtag&itmprp=enc%3AAQALAAAA8GfYFPkwiKCW4ZNSs2u11xDskEsSQCk2KNRZKmE7iVEB5Z1fMgIdbLD4BKz6OxyHhl3bvtCud9ghOxjjRDVS2nx4nztW8A3eHooDUD%2BTuraFJmjPRRxuq6UnGw3jfZEtxDuTwyYgyO1qFHfOQLprl90KKtzcn4--iRAIEbaUySusoQGuTwLKpG%2FnvVSXsNDa%2FEUqgcHlYYTey84Vhq1OxD5FLcuwDpDbOs0q0vlFikQHb7J7TOMj6AlJmJGKUrFaT7Z1adNgvnaXCXsrkA4oPlvllgOM4j69TuaRUs9IRuDFUEKtR57JsukzD4LwSVrdWQ%3D%3D%7Ctkp%3ABk9SR-6EscyaZw>Tyranids: Tyranid Prime With Lash Whip Warhammer 40K PRESALE 3/21</a>', '<span>$36.98</span>'], ['<img src=https://i.ebayimg.com/images/g/WBgAAOSw-LVkXVEC/s-l500.webp width=500>', '<a href=https://www.ebay.com/itm/183757561668?_skw=warhammer40k&epid=12055327819&itmmeta=01KKACC89CTYKFYQ5VYBJ9HA87&hash=item2ac8cddf44:g:WBgAAOSw-LVkXVEC&itmprp=enc%3AAQALAAAA8GfYFPkwiKCW4ZNSs2u11xDRzdQOFR0Yr4P%2Bj9

In [20]:
url = "http://localhost:8888/lab/tree/output.html"
webbrowser.open(url,new=2)

True